In [ ]:
# most recent particle file analyzer

findex = 143 #121 #120 #22 
step_choice = "first"

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}

my_steps = tstep_choices[step_choice]
boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"


In [ ]:
# Config Data

d = run.config_data(my_steps, "both")

In [ ]:
df = d.particles[2].to_dataframe()

ann_size(df)
add_gamma(df)

In [ ]:
roi = fq_cut(d.fields, 0, crect) 

In [ ]:
from matplotlib.path import Path

def choose_rect(df, rect):
    [x1, x2, y1, y2] = rect
    return df.loc[(x1 < df['x']) & (df['x'] < x2) & (y1 < df['y']) & (df['y'] < y2)]

def choose_circle(df, x0, y0, r):
    return df.loc[(df['x'] - x0) ** 2 + (df['y'] - y0) ** 2 < r ** 2]

def choose_circle_slice(df, x0, y0, r1, r2):
    df['dist'] = np.sqrt((df['x'] - x0) ** 2 + (df['y'] - y0) ** 2)

    return df.loc[(r1 < df['dist']) & (df['dist'] < r2)]

def choose_ellipse(df, x0, y0, a, b):
    return df.loc[((df['x'] - x0) / a) ** 2 + ((df['y'] - y0) / b) ** 2 < 1]

def choose_quad(df, points):

    # Create a Path object from the points
    path = Path(points)

    # Create an array of (x, y) points from the dataframe
    points_array = df[['x', 'y']].values

    # Use the Path object to determine which points are inside the quadrilateral
    mask = path.contains_points(points_array)

    # Return the dataframe rows that are inside the quadrilateral
    return df[mask]

In [ ]:

chosen_sets = [
    choose_rect(df, [0, 1000, 2500, 5500])
]

chosen_sets = [
    choose_quad(df, [(337, 1250), (420, 1250), (334, 1380), (417, 1380)])
]

colors = ['blue', 'red', 'green', 'pink', 'yellow', 'purple', 'orange', 'cyan', 'magenta', 'black']


In [ ]:
print(len(chosen_sets[0]))

In [ ]:
xdiff = 100
crect = [run.xlim / 2 - xdiff, run.xlim / 2 + xdiff, 0, run.ylim / 2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (10, 5))
fig.suptitle(run.boilerplate + f"\n t = {my_steps[0]}")

# Left Plot

for i in range(len(chosen_sets)):
    chosen = chosen_sets[i]
    alphas = [1, 1, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]
    ax1.scatter(chosen['x'], chosen['y'], s = 0.1, c = colors[i] , alpha = alphas[i])


plot_streamline(ax1, roi, roi.bx, roi.by, 0.75, 'w')
mesh = plot_fq3(ax1, roi, roi.dens / 16, "ola", vabsmax = 4)
#ax1.contour(cs.x, cs.y, chi, [0], colors = 'green')

bthresh = 0.9
ax1.contour(roi.x, roi.y, np.sqrt(bsq) / B0, [bthresh], colors = 'green')

ax1.set_xlim(crect[0], crect[1])
ax1.set_ylim(crect[2], crect[3])
ax1.set_aspect('equal')

cbar = fig.colorbar(mesh, ax = ax1)
cbar.set_label("$n / n_0$")

# Right Plot

log_bins = 10 ** np.linspace(-4, 4, 100) #300

for i in range(len(chosen_sets)):
    chosen = chosen_sets[i]
    add_gamma(chosen)
    chosen['beta_w'] = chosen['w'] / chosen['gamma']

    hist, bins = np.histogram((chosen['gamma'] - 1) / run.sigma, bins = log_bins, density = True)
    
    mean = np.mean(chosen['beta_w'])

    ax2.loglog(bins[1:], hist, color = colors[i], label = fr'{len(chosen):.2e} prtls ($\beta_w =$ {mean:.2f})')


ax2.set_xlabel("$(\gamma - 1) / \sigma$")
ax2.set_ylabel("$dN / d\gamma$")
ax2.set_title(f"Electron Spectra")
ax2.grid(visible = True)
ax2.axvline(1, color = 'black', linestyle = '--', label = '$\gamma - 1 = \sigma$')
ax2.axvline(1 / run.sigma, color = 'orange', linestyle = '--', label = '$\gamma - 1 = 1$')
#ax2.legend()
plt.tight_layout()

In [ ]:
print(0 / 0)

In [ ]:
# Initialize a list to store the average work done for each chosen set
average_work_done = []

# Loop through each chosen set
for chosen in chosen_sets:
    # Compute the work done on each particle by the electric field
    work_done = np.array(chosen['ex']) * np.array(chosen['u']) + np.array(chosen['ey']) * np.array(chosen['v']) + np.array(chosen['ez']) * np.array(chosen['w'])
    
    # Compute the average work done for the current set
    avg_work = np.mean(work_done)
    
    # Append the average work done to the list
    average_work_done.append(avg_work)

# Plot the results in a bar chart
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(len(chosen_sets)), average_work_done, color=colors[:len(chosen_sets)])
ax.set_xlabel('Chosen Set Index')
ax.set_ylabel('Average Work Done')
ax.set_title('Average Work Done on Each Particle by the Electric Field')
plt.show()

In [ ]:
# Define the rectangle for selecting particles

selected_particles = chosen_sets[0]
num_particles = len(selected_particles)

# Create a new subplot for the energy spectrum
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
fig.suptitle(f"Rectangle: {0}, Particles: {num_particles}\n" + run.boilerplate + f"\n t = {my_steps[0]}")

# Plot the u momentum spectrum
hist_u, bins_u = np.histogram(selected_particles['u'], bins=100, density=True)
ax1.plot(bins_u[:-1], hist_u, color='blue', label='u')

# Plot the v momentum spectrum
hist_v, bins_v = np.histogram(selected_particles['v'], bins=100, density=True)
ax1.plot(bins_v[:-1], hist_v, color='red', label='v')

# Plot the w momentum spectrum
hist_w, bins_w = np.histogram(selected_particles['w'], bins=100, density=True)
ax1.plot(bins_w[:-1], hist_w, color='green', label='w')

# Set labels and title for momentum spectra
ax1.set_xlabel('Momentum')
ax1.set_ylabel('$dN / d\gamma$')
ax1.set_yscale('log')
ax1.set_title('Momentum Spectra')
ax1.grid(True)
ax1.legend()

# Plot the energy spectrum
log_bins = 10 ** np.linspace(-4, 4, 100)
add_gamma(selected_particles)
selected_particles['beta_w'] = selected_particles['w'] / selected_particles['gamma']
hist, bins = np.histogram((selected_particles['gamma'] - 1) / run.sigma, bins=log_bins, density=True)
mean = np.mean(selected_particles['beta_w'])
average_gamma = selected_particles['gamma'].mean()

ax2.loglog(bins[1:], hist, color='purple', label=fr'{num_particles:.2e} prtls ($\beta_w =$ {mean:.2f})')
ax2.set_xlabel("$(\gamma - 1) / \sigma$")
ax2.set_ylabel("$dN / d\gamma$")
ax2.set_title(f"Electron Spectra (Average Gamma: {average_gamma:.2f})")
ax2.grid(visible=True)
ax2.axvline(1, color='black', linestyle='--', label='$\gamma - 1 = \sigma$')
ax2.axvline(1 / run.sigma, color='orange', linestyle='--', label='$\gamma - 1 = 1$')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
def lorentz_transform(u, v, w, beta_x, beta_y, beta_z):
    gamma = 1 / np.sqrt(1 - beta_x**2 - beta_y**2 - beta_z**2)
    u_prime = gamma * (u - beta_x * (gamma * u + beta_x * u + beta_y * v + beta_z * w))
    v_prime = gamma * (v - beta_y * (gamma * v + beta_x * u + beta_y * v + beta_z * w))
    w_prime = gamma * (w - beta_z * (gamma * w + beta_x * u + beta_y * v + beta_z * w))
    return u_prime, v_prime, w_prime

# Calculate the bulk flow velocity components

def compute_bulk_velocity(selected_particles):
    selected_particles['beta_x'] = selected_particles['u'] / selected_particles['gamma']
    selected_particles['beta_y'] = selected_particles['v'] / selected_particles['gamma']
    selected_particles['beta_z'] = selected_particles['w'] / selected_particles['gamma']

    bulk_beta_x = selected_particles['beta_x'].mean()
    bulk_beta_y = selected_particles['beta_y'].mean()
    bulk_beta_z = selected_particles['beta_z'].mean()
    return bulk_beta_x, bulk_beta_y, bulk_beta_z

bulk_beta_x, bulk_beta_y, bulk_beta_z = compute_bulk_velocity(selected_particles)


bulk_beta_x = 0
bulk_beta_y = -0.9 * 0.48
bulk_beta_z = 0


# Apply Lorentz transformation to the selected particles
selected_particles['u_prime'], selected_particles['v_prime'], selected_particles['w_prime'] = lorentz_transform(
    selected_particles['u'], selected_particles['v'], selected_particles['w'], -bulk_beta_x, -bulk_beta_y, -bulk_beta_z)
    #TODO not super sure about negative signs

# Calculate the new gamma' for each particle
selected_particles['gamma_prime'] = np.sqrt(1 + selected_particles['u_prime']**2 + selected_particles['v_prime']**2 + selected_particles['w_prime']**2)

# Compute the average gamma'
average_gamma_prime = selected_particles['gamma_prime'].mean()

# Create a new subplot for the energy spectrum after boosting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
fig.suptitle(f"Rectangle: {rect}, Particles: {num_particles}\n" +
             f"Bulk Flow Vector: ({bulk_u:.2e}, {bulk_v:.2e}, {bulk_w:.2e})\n" +
             run.boilerplate + f"\n t = {my_steps[0]}")

# Plot the u' momentum spectrum
hist_u_prime, bins_u_prime = np.histogram(selected_particles['u_prime'], bins=100, density=True)
ax1.plot(bins_u_prime[:-1], hist_u_prime, color='blue', label="u'")

# Plot the v' momentum spectrum
hist_v_prime, bins_v_prime = np.histogram(selected_particles['v_prime'], bins=100, density=True)
ax1.plot(bins_v_prime[:-1], hist_v_prime, color='red', label="v'")

# Plot the w' momentum spectrum
hist_w_prime, bins_w_prime = np.histogram(selected_particles['w_prime'], bins=100, density=True)
ax1.plot(bins_w_prime[:-1], hist_w_prime, color='green', label="w'")

# Set labels and title for momentum spectra
ax1.set_xlabel("Momentum")
ax1.set_ylabel("$dN / d\gamma$")
ax1.set_yscale("log")
ax1.set_title("Momentum Spectra in Bulk Flow Frame")
ax1.grid(True)
ax1.legend()

# Plot the energy spectrum after boosting
log_bins = 10 ** np.linspace(-4, 4, 100)
add_gamma(selected_particles)
selected_particles['beta_w_prime'] = selected_particles['w_prime'] / selected_particles['gamma']
hist_prime, bins_prime = np.histogram((selected_particles['gamma'] - 1) / run.sigma, bins=log_bins, density=True)
mean_prime = np.mean(selected_particles['beta_w_prime'])

ax2.loglog(bins_prime[1:], hist_prime, color='purple', label=fr'{num_particles:.2e} prtls ($\beta_w =$ {mean_prime:.2f})')
ax2.set_xlabel("$(\gamma - 1) / \sigma$")
ax2.set_ylabel("$dN / d\gamma$")
ax2.set_title("Electron Spectra in Bulk Flow Frame")
ax2.grid(visible=True)
ax2.axvline(1, color='black', linestyle='--', label='$\gamma - 1 = \sigma$')
ax2.axvline(1 / run.sigma, color='orange', linestyle='--', label='$\gamma - 1 = 1$')
ax2.legend()

# Add text annotation for average gamma'
ax2.text(0.05, 0.95, f"Average Gamma': {average_gamma_prime:.2f}", transform=ax2.transAxes, verticalalignment='top')

plt.tight_layout()
plt.show()


In [ ]:
# Create a new subplot for the energy spectrum
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
fig.suptitle(f"Rectangle: {rect}, Particles: {num_particles}\n" + run.boilerplate + f"\n t = {my_steps[0]}")

# Plot the u/gamma spectrum
hist_u, bins_u = np.histogram(selected_particles['u'] / selected_particles['gamma'], bins=100, density=True)
ax1.plot(bins_u[:-1], hist_u, color='blue', label='u/gamma')

# Plot the v/gamma spectrum
hist_v, bins_v = np.histogram(selected_particles['v'] / selected_particles['gamma'], bins=100, density=True)
ax1.plot(bins_v[:-1], hist_v, color='red', label='v/gamma')

# Plot the w/gamma spectrum
hist_w, bins_w = np.histogram(selected_particles['w'] / selected_particles['gamma'], bins=100, density=True)
ax1.plot(bins_w[:-1], hist_w, color='green', label='w/gamma')

# Set labels and title for momentum spectra
ax1.set_xlabel('Momentum / Gamma')
ax1.set_ylabel('$dN / d\gamma$')
ax1.set_yscale('log')
ax1.set_title('Momentum Spectra')
ax1.grid(True)
ax1.legend()

# Plot the energy spectrum
log_bins = 10 ** np.linspace(-4, 4, 100)
add_gamma(selected_particles)
selected_particles['beta_w'] = selected_particles['w'] / selected_particles['gamma']
hist, bins = np.histogram((selected_particles['gamma'] - 1) / run.sigma, bins=log_bins, density=True)
mean = np.mean(selected_particles['beta_w'])
average_gamma = selected_particles['gamma'].mean()

ax2.loglog(bins[1:], hist, color='purple', label=fr'{num_particles:.2e} prtls ($\beta_w =$ {mean:.2f})')
ax2.set_xlabel("$(\gamma - 1) / \sigma$")
ax2.set_ylabel("$dN / d\gamma$")
ax2.set_title(f"Electron Spectra (Average Gamma: {average_gamma:.2f})")
ax2.grid(visible=True)
ax2.axvline(1, color='black', linestyle='--', label='$\gamma - 1 = \sigma$')
ax2.axvline(1 / run.sigma, color='orange', linestyle='--', label='$\gamma - 1 = 1$')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
print(0 / 0)
# Define the gamma threshold
gamma_threshold = 2

# Filter particles with gamma over the threshold
particles_with_high_gamma = df[df['gamma'] > gamma_threshold]

# Create a scatter plot of the positions
plt.figure(figsize=(10, 6))
plt.scatter(particles_with_high_gamma['x'], particles_with_high_gamma['y'], s=0.1, c='blue')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Positions of Particles with Gamma > {gamma_threshold}')
plt.xlim([0, run.xlim])
plt.ylim([0, run.ylim])
plt.show()

In [ ]:
selected_particles = chosen_sets[0]
add_gamma(selected_particles)

# Filter particles based on the condition

filtered_particles = selected_particles[(selected_particles['gamma'] - 1) / run.sigma > elims[0]]
filtered_particles = filtered_particles[(filtered_particles['gamma'] - 1) / run.sigma < elims[1]]

# Define the resolution for the density grid
resolution = 50

# Create a 2D histogram to compute the number density
x_edges = np.linspace(filtered_particles['x'].min(), filtered_particles['x'].max(), resolution)
y_edges = np.linspace(filtered_particles['y'].min(), filtered_particles['y'].max(), resolution)
density, x_edges, y_edges = np.histogram2d(filtered_particles['x'], filtered_particles['y'], bins=[x_edges, y_edges])

# Plot the number density
fig, ax = plt.subplots(figsize=(8, 6))
c = ax.pcolormesh(x_edges, y_edges, density.T, cmap='viridis')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Count (elims = {elims[0], elims[1]})')
ax.set_aspect('equal')  # Set aspect ratio to 1
fig.colorbar(c, ax=ax, label='Number Density')
plt.show()

#def plot_fq2(plot, field_data, quantity, title, vabsmax):
    # Plot colormesh with fixed endpoints

#    mesh = plot.pcolormesh(field_data.x, field_data.y, quantity, cmap = 'gray', vmax = vabsmax, vmin = 0)
#    plot.set_xlabel('$c/\omega_{pe}$'); plot.set_ylabel('$c/\omega_{pe}$')
#    plot.set_title(f"{title}")
#    return mesh

In [ ]:
chosen_sets = [choose_circle_slice(df, 60, 69, 0, 15),
                choose_rect(df, [58.6, 61.4, 105, 125]),
                choose_rect(df, [56, 65, 85, 100])] #10
 
chosen_sets = [choose_circle_slice(df, 60, 73.3, 0, 4),
                choose_rect(df, [59, 61, 78, 90]),
                choose_rect(df, [59, 61, 116, 121]),
                choose_circle_slice(df, 60, 123, 0, 1)]

chosen_sets = [
    choose_rect(df, [59.3, 60.7, 105, 135]) 
]

chosen_sets = [
    choose_rect(df, [59.3, 60.7, 105, 120]) 
]

chosen_sets = [
    choose_circle_slice(df, 120, 410, 0, 40) 
]

chosen_sets = [
    choose_ellipse(df, 120, 410, 40, 60) 
]

chosen_sets = [
    choose_rect(df, [100, 140, 200, 400]),
    choose_rect(df, [0, run.xlim, 0, run.ylim])
]

chosen_sets = [
    choose_rect(df, [100, 140, 400, 450]),
    choose_rect(df, [100, 140, 350, 400]),
    choose_rect(df, [100, 140, 300, 350]),
    choose_rect(df, [100, 140, 250, 300]),
    choose_rect(df, [100, 140, 200, 250]),
    choose_rect(df, [100, 140, 150, 200]),
    choose_rect(df, [100, 140, 100, 150]),
]

chosen_sets = [
    choose_rect(df, [100, 140, 150, 300]),
]

chosen_sets = [
    choose_rect(df, [110, 130, 150, 300]),
]

chosen_sets = [
    choose_rect(df, [117.5, 122.5, 200, 250]),
    choose_rect(df, [110, 115, 200, 250])
]

#chosen_sets = [choose_circle_slice(df, 60, 168.5, 0, 5),
#               choose_rect(df, [59, 61, 175, 200])]

chosen_sets = [
    choose_rect(df, [374, 376, 1500, 1515]),
    choose_rect(df, [110, 115, 200, 250])
]

chosen_sets = [
    choose_rect(df, [59, 61, 117, 124])
]

chosen_sets = [
    choose_rect(df, [365, 380, 300, 350]),
    choose_rect(df, [335, 345, 300, 350]),
    choose_rect(df, [315, 325, 300, 350]),
    choose_rect(df, [200, 220, 300, 350])
]
